# Julia Machine Learning demo

## My first neural network

Before running this notebook, start Julia in the project folder with `julia --project=.` and add the required packages in Pkg mode with:

```julia
add CSV DataFrames Lux Optimisers Zygote Statistics Random
```

`Printf` is part of Julia's standard library, so it does not need to be added.

In this demo, we train a small neural network that predicts a sauna enjoyment score from three inputs:

- sauna temperature
- outside temperature
- minutes since the coffee was brewed

The goal is to learn the relationship from example data and then use the trained model to score a few Finnish sauna scenarios.

## Step 1: Load and prepare the data

We already have example sauna data in a text file, so the first job is to load it into Julia and turn it into numeric arrays the network can use.

In [1]:
using CSV
using DataFrames
using Lux
using Random
using Optimisers
using Zygote
using Statistics
using Printf

sauna_path = joinpath(pwd(), "data", "sauna_log.txt")
sauna_log = CSV.read(sauna_path, DataFrame)

# Convert the table into a numeric matrix.
# Rows 1:3 will be inputs and row 4 will be the target score.
feature_table = Matrix(sauna_log[:, [:sauna_temp, :outside_temp, :coffee_min, :score]])
data = Float32.(permutedims(feature_table))

input_data_raw = data[1:3, :]
target_scores = data[4:4, :]

# We normalize the inputs because the three features live on very different
# scales. This makes training more stable and helps gradient-based updates.
feature_mean = mean(input_data_raw, dims = 2)
feature_std = std(input_data_raw, dims = 2)
input_data = (input_data_raw .- feature_mean) ./ feature_std

# Split to train and test (80/20).
# The model learns only from the training split. The test split is kept aside
# and used only at the end to measure performance on unseen data.
rng = Xoshiro(123)
n_train = Int(round(0.8 * size(input_data, 2)))
shuffled_indices = randperm(rng, size(input_data, 2))

X_train = input_data[:, shuffled_indices[1:n_train]]
Y_train = target_scores[:, shuffled_indices[1:n_train]]
X_test = input_data[:, shuffled_indices[n_train + 1:end]]
Y_test = target_scores[:, shuffled_indices[n_train + 1:end]]

@printf "Train: %d  Test: %d\n" size(X_train, 2) size(X_test, 2)

Train: 160  Test: 40


## Step 2: Build the model

Now we define a very small neural network. It takes 3 inputs, transforms them through a hidden layer, and returns 1 predicted enjoyment score.

```
Inputs              Hidden layer               Output

sauna temp    o ----\
                  o --\
outside temp  o -----)--- o  o  o  o  o  o  o  o ----> o   predicted score
                  o --/
coffee age    o ----/

3 features        Dense(3 => 8, tanh)      Dense(8 => 1, sigmoid)
```

The final sigmoid keeps the prediction in the 0 to 1 range.

In [2]:
model = Chain(
    Dense(3 => 8, tanh),
    Dense(8 => 1, sigmoid)
)

# Initialize model parameters and state
parameters, model_state = Lux.setup(rng, model)

((layer_1 = (weight = Float32[-0.71197426 0.5440084 -0.9027876; -0.62075335 -0.5090574 1.4643831; … ; 0.7166022 -0.17811993 0.5175801; -1.5047655 -0.24505515 1.5166628], bias = Float32[0.23182774, -0.18704338, 0.17974052, 0.15094592, -0.023374384, 0.47140634, -0.18232994, 0.08013492]), layer_2 = (weight = Float32[-0.48614922 0.6079607 … -0.5020537 -0.45970005], bias = Float32[0.24544002])), (layer_1 = NamedTuple(), layer_2 = NamedTuple()))

## Step 3: Train the network

We now define the loss and the training function. The training function takes the network, its parameters, and the training data, then repeatedly updates the parameters to reduce prediction error.

- Epochs = how many times we loop over the training data
- Learning rate = how large each parameter update should be

In [3]:
# This function takes a batch of inputs and target scores, runs the network,
# and returns the mean squared error. Training will try to make this smaller.
function mse_loss(model, parameters, model_state, batch)
    x, y_true = batch
    y_pred, new_model_state = model(x, parameters, model_state)
    loss = mean((y_pred .- y_true) .^ 2)
    return loss, new_model_state, ()
end

function train_model(model, parameters, model_state, train_batch; epochs = 500, lr = 0.01f0)
    train_state = Training.TrainState(model, parameters, model_state, Adam(lr))
    for epoch in 1:epochs
        _, loss, _, train_state = Training.single_train_step!(AutoZygote(), mse_loss, train_batch, train_state)
        @printf "Epoch %4d  Training loss = %.6f\n" epoch loss
    end
    return train_state
end

train_state = train_model(model, parameters, model_state, (X_train, Y_train))

Epoch    1  Training loss = 0.105149
Epoch    2  Training loss = 0.103422
Epoch    3  Training loss = 0.102009
Epoch    4  Training loss = 0.100871
Epoch    5  Training loss = 0.099936
Epoch    6  Training loss = 0.099146
Epoch    7  Training loss = 0.098451
Epoch    8  Training loss = 0.097809
Epoch    9  Training loss = 0.097179
Epoch   10  Training loss = 0.096531
Epoch   11  Training loss = 0.095855
Epoch   12  Training loss = 0.095156
Epoch   13  Training loss = 0.094442
Epoch   14  Training loss = 0.093724
Epoch   15  Training loss = 0.093007
Epoch   16  Training loss = 0.092288
Epoch   17  Training loss = 0.091562
Epoch   18  Training loss = 0.090823
Epoch   19  Training loss = 0.090062
Epoch   20  Training loss = 0.089277
Epoch   21  Training loss = 0.088464
Epoch   22  Training loss = 0.087621
Epoch   23  Training loss = 0.086749
Epoch   24  Training loss = 0.085846
Epoch   25  Training loss = 0.084914
Epoch   26  Training loss = 0.083954
Epoch   27  Training loss = 0.082967
E

TrainState(
    Chain(
        layer_1 = Dense(3 => 8, tanh),            # 32 parameters
        layer_2 = Dense(8 => 1, σ),               # 9 parameters
    ),
    number of parameters: 41
    number of states: 0
    optimizer: Adam(eta=0.01, beta=(0.9, 0.999), epsilon=1.0e-8)
    step: 500
)


## Step 4: Evaluate on the test split

After training, run the model on the held-out data to get a final test error that summarizes how well it predicts unseen examples.

In [4]:
Y_pred_test, _ = model(X_test, train_state.parameters, train_state.states)
test_mse = mean((Y_pred_test .- Y_test) .^ 2)
test_mae = mean(abs.(Y_pred_test .- Y_test))

@printf "Test MSE: %.6f  Test MAE: %.4f\n" test_mse test_mae

Test MSE: 0.006914  Test MAE: 0.0638


## Step 5: Predict a few sauna scenarios

Finally, try a few hand-picked sauna situations and see what score the trained network gives to each one.

In [5]:
scenarios = [
    (sauna = 80, outside = -20, coffee = 10),
    (sauna = 65, outside = 5, coffee = 100),
    (sauna = 95, outside = -25, coffee = 30)
]

# This function takes one scenario, normalizes it the same way as the training
# data, feeds it through the trained network, and returns one predicted score.
function predict_score(s, model, parameters, model_state, feature_mean, feature_std)
    x = Float32.([
        (s.sauna - feature_mean[1]) / feature_std[1],
        (s.outside - feature_mean[2]) / feature_std[2],
        (s.coffee - feature_mean[3]) / feature_std[3]
    ])
    pred, _ = model(reshape(x, 3, 1), parameters, model_state)
    return pred[1]
end

for scenario in scenarios
    predicted_score = predict_score(scenario, model, train_state.parameters, train_state.states, feature_mean, feature_std)
    @printf "Scenario: sauna=%dC outside=%dC coffee=%dmin\n" scenario.sauna scenario.outside scenario.coffee
    @printf "Predicted score: %.2f\n\n" predicted_score
end

Scenario: sauna=80C outside=-20C coffee=10min
Predicted score: 0.98

Scenario: sauna=65C outside=5C coffee=100min
Predicted score: 0.10

Scenario: sauna=95C outside=-25C coffee=30min
Predicted score: 0.40

